In [1]:
# Reconcile NB05 vs NB08 on momentum only (single strategy, single variable at a time)
import pandas as pd, numpy as np

# 1. Make sure both engines export gross_ret, net_ret, turnover, cost separately.
#    If NB05 only saves net, re-run it with gross output added.
nb05 = pd.read_parquet('data/working/results_nb05_momentum.parquet')
nb08 = pd.read_parquet('data/working/results_nb08_momentum.parquet')

# 2. Align on month-start
for df in (nb05, nb08):
    df.index = pd.to_datetime([str(p) for p in df.index.to_period('M')])

merged = nb05.join(nb08, lsuffix='_05', rsuffix='_08').dropna()

# 3. Decompose the difference
merged['gross_diff']    = merged['gross_ret_08']  - merged['gross_ret_05']
merged['cost_diff']     = merged['cost_08']       - merged['cost_05']
merged['turnover_diff'] = merged['turnover_08']   - merged['turnover_05']

print("Cumulative gross diff:", merged['gross_diff'].sum())
print("Cumulative cost  diff:", merged['cost_diff'].sum())

# 4. If gross diff is non-zero, find the worst months and inspect holdings
worst = merged['gross_diff'].abs().nlargest(10)
print(worst)
# For each worst month, dump the two portfolios and diff the ticker sets:
# missing_in_08 = set(holdings_05[m]) - set(holdings_08[m])
# new_in_08     = set(holdings_08[m]) - set(holdings_05[m])

FileNotFoundError: [Errno 2] No such file or directory: 'data/working/results_nb05_momentum.parquet'

In [2]:
import os
from pathlib import Path

working = Path('data/working')
print("=== Files in data/working/ ===")
for f in sorted(working.glob('*.parquet')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s}  {size_kb:>8.1f} KB")

print("\n=== Files in data/ (top level) ===")
for f in sorted(Path('data').glob('*')):
    if f.is_file():
        print(f"  {f.name}")

=== Files in data/working/ ===

=== Files in data/ (top level) ===


In [3]:
import os
from pathlib import Path
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
print("cwd:", os.getcwd())

cwd: C:\Users\user\ftse250_study


In [4]:
import pandas as pd
from pathlib import Path

working = Path('data/working')

# Inspect every backtest_* and results_* file
files_to_check = sorted(working.glob('backtest_*.parquet')) + \
                 sorted(working.glob('results_*.parquet')) + \
                 sorted(working.glob('holdout_*.parquet'))

for f in files_to_check:
    print(f"\n=== {f.name} ===")
    df = pd.read_parquet(f)
    print(f"  shape:  {df.shape}")
    print(f"  index:  {df.index.name} | dtype: {df.index.dtype}")
    if len(df) > 0:
        print(f"  range:  {df.index.min()}  →  {df.index.max()}")
    print(f"  cols:   {list(df.columns)}")
    print(f"  dtypes:")
    for c, d in df.dtypes.items():
        print(f"            {c:30s}  {d}")
    print(f"  head:")
    print(df.head(3).to_string().replace('\n', '\n          '))


=== backtest_low-vol.parquet ===
  shape:  (161, 10)
  index:  None | dtype: int64
  range:  0  →  160
  cols:   ['formation_month', 'holding_month', 'port_ret', 'n_stocks', 'n_with_returns', 'portfolio', 'turnover', 'spread_cost', 'tc_monthly', 'port_ret_net']
  dtypes:
            formation_month                 datetime64[us]
            holding_month                   datetime64[us]
            port_ret                        float64
            n_stocks                        int64
            n_with_returns                  int64
            portfolio                       object
            turnover                        float64
            spread_cost                     float64
            tc_monthly                      float64
            port_ret_net                    float64
  head:
  formation_month holding_month  port_ret  n_stocks  n_with_returns                                                                                                                           

In [5]:
# === Diagnostic 1: NB05 alpha (gross vs net), training period ===
import os
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

import pandas as pd, numpy as np
import statsmodels.api as sm

TRAIN_START = '2010-08-01'
TRAIN_END   = '2023-12-31'

def to_ms(idx):
    return pd.to_datetime([str(p) for p in pd.DatetimeIndex(idx).to_period('M')])

# --- Inspect the long-format files first ---
b_raw = pd.read_parquet('data/working/benchmark_train.parquet')
s_raw = pd.read_parquet('data/working/sonia_train.parquet')
print("Benchmark fields:", sorted(b_raw['field'].unique()))
print("SONIA fields:    ", sorted(s_raw['field'].unique()))
print("SONIA value range:", s_raw['value'].min(), "→", s_raw['value'].max(),
      "(if ~0–6, it's annualised %; if ~0.0–0.06, it's decimal)")

# --- Build monthly benchmark return ---
b = b_raw.copy()
b['date'] = pd.to_datetime(b['date'])
for fld in ['TOT_RETURN_INDEX_GROSS_DVDS', 'PX_LAST']:
    if fld in b['field'].unique():
        b = b[b['field'] == fld].set_index('date')['value'].sort_index()
        print(f"Benchmark using: {fld}")
        break
b_m = b.resample('ME').last()
b_m.index = to_ms(b_m.index)
mkt_ret = b_m.pct_change().rename('mkt_ret')

# --- Build monthly RF from SONIA ---
s = s_raw.copy()
s['date'] = pd.to_datetime(s['date'])
fld = s['field'].value_counts().idxmax()  # most common field
print(f"SONIA using: {fld}")
s = s[s['field'] == fld].set_index('date')['value'].sort_index()
# Detect units: if values mostly > 1, treat as annualised %
scale = 100.0 if s.median() > 1 else 1.0
s_m = s.resample('ME').mean() / scale / 12.0     # decimal monthly RF
s_m.index = to_ms(s_m.index)
rf = s_m.rename('rf')

# --- NW(6) regression helper ---
def alpha_nw(port_ret, mkt, rf, lags=6):
    df = pd.concat([port_ret.rename('p'), mkt.rename('m'), rf.rename('rf')],
                   axis=1).dropna()
    y = df['p'] - df['rf']
    x = sm.add_constant(df['m'] - df['rf'])
    m = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return {
        'alpha_ann_%': m.params['const'] * 12 * 100,
        't_alpha':     m.tvalues['const'],
        'beta':        m.params.iloc[1],
        'sharpe_ann':  (y.mean() / y.std()) * np.sqrt(12),
        'n_months':    int(len(df)),
        'mean_ret_ann_%': df['p'].mean() * 12 * 100,
    }

# --- Run on NB05 momentum and low-vol ---
files = {'momentum': 'data/working/backtest_momentum.parquet',
         'low-vol':  'data/working/backtest_low-vol.parquet'}

rows = []
for name, path in files.items():
    bt = pd.read_parquet(path)
    bt['holding_month'] = pd.to_datetime(bt['holding_month'])
    bt = bt[(bt['holding_month'] >= TRAIN_START) &
            (bt['holding_month'] <= TRAIN_END)].copy()
    bt = bt.set_index('holding_month'); bt.index = to_ms(bt.index)
    for kind, col in [('gross', 'port_ret'), ('net', 'port_ret_net')]:
        r = alpha_nw(bt[col], mkt_ret, rf)
        r['strategy'] = name; r['returns'] = kind
        # Cost drag:
        r['cost_drag_ann_%'] = (bt['port_ret'] - bt['port_ret_net']).mean() * 12 * 100
        rows.append(r)

out = pd.DataFrame(rows).set_index(['strategy', 'returns'])
out = out[['mean_ret_ann_%', 'alpha_ann_%', 't_alpha', 'beta',
           'sharpe_ann', 'cost_drag_ann_%', 'n_months']]
pd.set_option('display.float_format', '{:.3f}'.format)
print("\n=== NB05 baseline alphas (training 2010-08 to 2023-12) ===")
print(out)

Benchmark fields: ['PX_LAST', 'TOT_RETURN_INDEX_GROSS_DVDS']
SONIA fields:     ['PX_LAST']
SONIA value range: 0.0394 → 5.1889 (if ~0–6, it's annualised %; if ~0.0–0.06, it's decimal)
Benchmark using: TOT_RETURN_INDEX_GROSS_DVDS
SONIA using: PX_LAST

=== NB05 baseline alphas (training 2010-08 to 2023-12) ===
                  mean_ret_ann_%  alpha_ann_%  t_alpha  beta  sharpe_ann  \
strategy returns                                                           
momentum gross            14.282        6.092    2.728 1.011      -1.699   
         net              11.591        3.380    1.516 1.010      -1.770   
low-vol  gross             8.897       -4.041   -2.011 0.943      -1.970   
         net               7.894       -5.091   -2.524 0.942      -2.000   

                  cost_drag_ann_%  n_months  
strategy returns                             
momentum gross              2.691       161  
         net                2.691       161  
low-vol  gross              1.003       161  
    

C:\Users\user\AppData\Local\Temp\ipykernel_9120\2814750675.py:48: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([port_ret.rename('p'), mkt.rename('m'), rf.rename('rf')],
C:\Users\user\AppData\Local\Temp\ipykernel_9120\2814750675.py:48: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([port_ret.rename('p'), mkt.rename('m'), rf.rename('rf')],
C:\Users\user\AppData\Local\Temp\ipykernel_9120\2814750675.py:48: Pandas4Warning: Sorting by default when concatenating all Da

In [6]:
# === Diagnostic 1 (corrected SONIA scale) ===
import os; os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import pandas as pd, numpy as np, statsmodels.api as sm

TRAIN_START, TRAIN_END = '2010-08-01', '2023-12-31'

def to_ms(idx):
    return pd.to_datetime([str(p) for p in pd.DatetimeIndex(idx).to_period('M')])

# --- Benchmark ---
b = pd.read_parquet('data/working/benchmark_train.parquet')
b['date'] = pd.to_datetime(b['date'])
b = b[b['field'] == 'TOT_RETURN_INDEX_GROSS_DVDS'].set_index('date')['value'].sort_index()
b_m = b.resample('ME').last()
b_m.index = to_ms(b_m.index)
mkt_ret = b_m.pct_change().rename('mkt_ret')

# --- SONIA: ALWAYS annualised %, no auto-detect ---
s = pd.read_parquet('data/working/sonia_train.parquet')
s['date'] = pd.to_datetime(s['date'])
s = s[s['field'] == 'PX_LAST'].set_index('date')['value'].sort_index()
s_m = s.resample('ME').mean() / 100.0 / 12.0   # % -> decimal -> monthly
s_m.index = to_ms(s_m.index)
rf = s_m.rename('rf')

# Sanity check
print(f"RF monthly mean: {rf.mean():.5f}  (should be ~0.0008–0.002)")
print(f"RF monthly max:  {rf.max():.5f}    (should be ~0.004 in late 2023)")
print(f"RF monthly min:  {rf.min():.5f}    (should be ~0.00003)")

def alpha_nw(port_ret, mkt, rf, lags=6):
    df = pd.concat([port_ret.rename('p'), mkt.rename('m'), rf.rename('rf')],
                   axis=1, sort=True).dropna()
    y = df['p'] - df['rf']
    x = sm.add_constant(df['m'] - df['rf'])
    m = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return {
        'alpha_ann_%':    m.params['const'] * 12 * 100,
        't_alpha':        m.tvalues['const'],
        'beta':           m.params.iloc[1],
        'sharpe_ann':     (y.mean() / y.std()) * np.sqrt(12),
        'mean_ret_ann_%': df['p'].mean() * 12 * 100,
        'n_months':       int(len(df)),
    }

files = {'momentum': 'data/working/backtest_momentum.parquet',
         'low-vol':  'data/working/backtest_low-vol.parquet'}

rows = []
for name, path in files.items():
    bt = pd.read_parquet(path)
    bt['holding_month'] = pd.to_datetime(bt['holding_month'])
    bt = bt[(bt['holding_month'] >= TRAIN_START) &
            (bt['holding_month'] <= TRAIN_END)].copy()
    bt = bt.set_index('holding_month'); bt.index = to_ms(bt.index)
    for kind, col in [('gross', 'port_ret'), ('net', 'port_ret_net')]:
        r = alpha_nw(bt[col], mkt_ret, rf)
        r['strategy'] = name; r['returns'] = kind
        r['cost_drag_ann_%'] = (bt['port_ret'] - bt['port_ret_net']).mean() * 12 * 100
        rows.append(r)

out = pd.DataFrame(rows).set_index(['strategy', 'returns'])
out = out[['mean_ret_ann_%', 'alpha_ann_%', 't_alpha', 'beta',
          'sharpe_ann', 'cost_drag_ann_%', 'n_months']]
pd.set_option('display.float_format', '{:.3f}'.format)
print("\n=== NB05 baseline alphas (training 2010-08 to 2023-12) ===")
print(out)

RF monthly mean: 0.00064  (should be ~0.0008–0.002)
RF monthly max:  0.00432    (should be ~0.004 in late 2023)
RF monthly min:  0.00004    (should be ~0.00003)

=== NB05 baseline alphas (training 2010-08 to 2023-12) ===
                  mean_ret_ann_%  alpha_ann_%  t_alpha  beta  sharpe_ann  \
strategy returns                                                           
momentum gross            14.282        5.852    3.310 0.939       0.871   
         net              11.591        3.166    1.787 0.938       0.698   
low-vol  gross             8.897        3.089    3.124 0.616       0.825   
         net               7.894        2.090    2.112 0.616       0.723   

                  cost_drag_ann_%  n_months  
strategy returns                             
momentum gross              2.691       161  
         net                2.691       161  
low-vol  gross              1.003       161  
         net                1.003       161  


In [7]:
# === Diagnostic 2: NB05 holdings sanity & turnover profile ===
import os; os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import pandas as pd, numpy as np

for name in ['momentum', 'low-vol']:
    bt = pd.read_parquet(f'data/working/backtest_{name}.parquet')
    bt['holding_month'] = pd.to_datetime(bt['holding_month'])
    bt = bt[(bt['holding_month'] >= '2010-08-01') &
            (bt['holding_month'] <= '2023-12-31')].copy()

    print(f"\n=== {name.upper()} (NB05) ===")
    print(f"  Months:               {len(bt)}")
    print(f"  Avg # stocks:         {bt['n_stocks'].mean():.1f}  "
          f"(min {bt['n_stocks'].min()}, max {bt['n_stocks'].max()})")
    print(f"  Avg monthly turnover: {bt['turnover'].mean():.3f}  "
          f"(steady-state ex month-1: {bt['turnover'].iloc[1:].mean():.3f})")
    print(f"  Avg spread cost:      {bt['spread_cost'].mean():.5f}  per month")
    print(f"  Avg total tc:         {bt['tc_monthly'].mean():.5f}  per month")
    print(f"  Implied stamp duty:   {(bt['tc_monthly'] - bt['spread_cost']).mean():.5f}  per month")
    print(f"  Stamp/turnover ratio: "
          f"{((bt['tc_monthly'] - bt['spread_cost']).mean() / bt['turnover'].mean()):.4f}  "
          "(should be 0.005 if charged on all turnover, 0.0025 if buys-only)")

# Also: what does NB05 do with the 'plus_vol' files? Are they still being used somewhere?
# Compare turnover NB05 vanilla vs plus_vol — the volume filter probably hurts via turnover
print("\n=== Volume-filter impact on turnover (NB05) ===")
for name in ['momentum', 'low-vol', 'reversal']:
    a = pd.read_parquet(f'data/working/backtest_{name}.parquet')['turnover'].mean()
    b = pd.read_parquet(f'data/working/backtest_{name}_plus_vol.parquet')['turnover'].mean()
    print(f"  {name:10s} vanilla {a:.3f} → plus_vol {b:.3f}  (Δ {b-a:+.3f})")


=== MOMENTUM (NB05) ===
  Months:               161
  Avg # stocks:         49.7  (min 47, max 51)
  Avg monthly turnover: 0.251  (steady-state ex month-1: 0.246)
  Avg spread cost:      0.00388  per month
  Avg total tc:         0.00224  per month
  Implied stamp duty:   -0.00164  per month
  Stamp/turnover ratio: -0.0065  (should be 0.005 if charged on all turnover, 0.0025 if buys-only)

=== LOW-VOL (NB05) ===
  Months:               161
  Avg # stocks:         49.9  (min 49, max 50)
  Avg monthly turnover: 0.118  (steady-state ex month-1: 0.113)
  Avg spread cost:      0.00202  per month
  Avg total tc:         0.00084  per month
  Implied stamp duty:   -0.00119  per month
  Stamp/turnover ratio: -0.0101  (should be 0.005 if charged on all turnover, 0.0025 if buys-only)

=== Volume-filter impact on turnover (NB05) ===
  momentum   vanilla 0.251 → plus_vol 0.686  (Δ +0.435)
  low-vol    vanilla 0.118 → plus_vol 0.616  (Δ +0.498)
  reversal   vanilla 0.770 → plus_vol 0.865  (Δ +0.095

In [8]:
import os; os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
from pathlib import Path

for f in sorted(Path('.').glob('*.ipynb')):
    print(f"  {f.name}  ({f.stat().st_size//1024} KB)")

# Also check for a notebooks/ subdir
for sub in ['notebooks', 'nb', 'src']:
    p = Path(sub)
    if p.exists():
        print(f"\n  {sub}/")
        for f in sorted(p.glob('*.ipynb')):
            print(f"    {f.name}  ({f.stat().st_size//1024} KB)")


  notebooks/
    01_membership_pull.ipynb  (22 KB)
    02_price_pull.ipynb  (14 KB)
    03_data_verification.ipynb  (14 KB)
    04_signal_computation.ipynb  (18 KB)
    05_backtestest_engine.ipynb  (307 KB)
    06_holdout_test.ipynb  (188 KB)
    07_combined_portfolio.ipynb  (567 KB)
    08_robustness_checks.ipynb  (396 KB)
    09_factor_attribution.ipynb  (185 KB)
    10_reconciliation.ipynb  (78 KB)


In [9]:
import os
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

In [10]:
# === Diagnostic 3b v2: shift=0 (NB08 default) vs shift=1 (NB05 convention) ===
import os; os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import pandas as pd, numpy as np, statsmodels.api as sm

TRAIN_START, TRAIN_END = '2010-08-01', '2023-12-31'
STAMP_DUTY = 0.005

def to_ms(idx):
    """Force any datetime-like index → month-start datetime64[ns]."""
    return pd.PeriodIndex(pd.to_datetime(idx), freq='M').to_timestamp()

# --- Load ---
monthly = pd.read_parquet('data/working/monthly_signals.parquet')
monthly['month'] = pd.to_datetime(monthly['month'])
monthly = monthly.sort_values(['ticker','month']).reset_index(drop=True)

cs = pd.read_parquet('data/working/monthly_cs_spread.parquet')
cs['month'] = pd.to_datetime(cs['month'])

# Build z_composite if not stored
if 'z_composite' not in monthly.columns:
    g = monthly.groupby('month')
    monthly['z_mom'] = g['momentum_12_1'].transform(lambda x: (x - x.mean())/x.std())
    monthly['z_vol'] = g['vol_90d'].transform(lambda x: -(x - x.mean())/x.std())
    monthly['z_composite'] = (monthly['z_mom'] + monthly['z_vol']) / 2

# Benchmark + RF
b = pd.read_parquet('data/working/benchmark_train.parquet')
b['date'] = pd.to_datetime(b['date'])
b = b[b['field']=='TOT_RETURN_INDEX_GROSS_DVDS'].set_index('date')['value'].sort_index()
mkt = b.resample('ME').last().pct_change()
mkt.index = to_ms(mkt.index); mkt = mkt.rename('mkt')

s = pd.read_parquet('data/working/sonia_train.parquet')
s['date'] = pd.to_datetime(s['date'])
s = s[s['field']=='PX_LAST'].set_index('date')['value'].sort_index()
rf = s.resample('ME').mean() / 100.0 / 12.0
rf.index = to_ms(rf.index); rf = rf.rename('rf')

# --- Engine (NB08-equivalent) parametrised by extra signal shift ---
def run_bt(signals, signal_col, long_top, q=0.20, extra_shift=0):
    df = signals.copy()
    if extra_shift:
        df[signal_col] = df.groupby('ticker')[signal_col].shift(extra_shift)
    df = df.dropna(subset=[signal_col, 'monthly_ret'])
    df = df[(df['month'] >= TRAIN_START) & (df['month'] <= TRAIN_END)]
    out, prev = [], set()
    for mo in sorted(df['month'].unique()):
        md = df[df['month']==mo]
        if len(md) < 20: continue
        if long_top:
            thr = md[signal_col].quantile(1-q); sel = md[md[signal_col] >= thr]
        else:
            thr = md[signal_col].quantile(q);   sel = md[md[signal_col] <= thr]
        if sel.empty: continue
        cur = set(sel['ticker'])
        gross = sel['monthly_ret'].mean()
        buys, sells = cur - prev, prev - cur
        msp = cs[cs['month']==mo].set_index('ticker')['spread']
        def avs(t):
            if not t: return 0.0
            f = msp.reindex(list(t)).dropna()
            return f.mean() if len(f) else (msp.mean() if len(msp) else 0.0)
        n = len(cur); bw, sw = len(buys)/n, len(sells)/n
        hb = avs(buys)/2  if buys  else 0
        hs = avs(sells)/2 if sells else 0
        tc = bw*(hb + STAMP_DUTY) + sw*hs
        out.append({'month': mo, 'gross': gross, 'net': gross-tc, 'tc': tc, 'n': n})
        prev = cur
    bt = pd.DataFrame(out).set_index('month')
    if not bt.empty:
        bt.index = to_ms(bt.index)
    return bt

def alpha_nw(series, lags=6):
    s = series.copy()
    s.index = to_ms(s.index)                              # belt-and-braces
    df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()
    if len(df) < 10:
        return {'alpha%': np.nan, 't': np.nan, 'beta': np.nan,
                'sharpe': np.nan, 'mean%': np.nan, 'n_mo': len(df)}
    y = df['p'] - df['rf']
    X = sm.add_constant(df['mkt'] - df['rf'])
    m = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return {'alpha%': m.params['const']*12*100,
            't':      m.tvalues['const'],
            'beta':   m.params.iloc[1],
            'sharpe': (y.mean()/y.std())*np.sqrt(12),
            'mean%':  df['p'].mean()*12*100,
            'n_mo':   len(df)}

configs = [('momentum_12_1', True), ('vol_90d', False), ('z_composite', True)]

rows = []
for sig, lt in configs:
    if sig not in monthly.columns:
        print(f"  ! {sig} missing"); continue
    for shift in (0, 1):
        bt = run_bt(monthly, sig, long_top=lt, q=0.20, extra_shift=shift)
        if bt.empty:
            print(f"  ! {sig} shift={shift} produced empty"); continue
        print(f"  {sig:15s} shift={shift}  →  {len(bt)} months, avg n={bt['n'].mean():.1f}")
        for kind in ('gross','net'):
            r = alpha_nw(bt[kind])
            r.update({'signal': sig, 'shift': shift, 'kind': kind})
            rows.append(r)

out = pd.DataFrame(rows).set_index(['signal','shift','kind'])
out = out[['mean%','alpha%','t','beta','sharpe','n_mo']]
pd.set_option('display.float_format', '{:.3f}'.format)
print("\n=== shift=0 (NB08 as-is) vs shift=1 (extra lag → NB05 convention) ===")
print(out)

  momentum_12_1   shift=0  →  161 months, avg n=89.6


C:\Users\user\AppData\Local\Temp\ipykernel_9120\600414954.py:78: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_9120\600414954.py:78: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()


  momentum_12_1   shift=1  →  161 months, avg n=89.4


C:\Users\user\AppData\Local\Temp\ipykernel_9120\600414954.py:78: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_9120\600414954.py:78: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()


  vol_90d         shift=0  →  161 months, avg n=91.8


...

C:\Users\user\AppData\Local\Temp\ipykernel_9120\600414954.py:78: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_9120\600414954.py:78: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()


  z_composite     shift=0  →  161 months, avg n=89.6


C:\Users\user\AppData\Local\Temp\ipykernel_9120\600414954.py:78: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_9120\600414954.py:78: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()


  z_composite     shift=1  →  161 months, avg n=89.3

=== shift=0 (NB08 as-is) vs shift=1 (extra lag → NB05 convention) ===
                           mean%  alpha%     t  beta  sharpe  n_mo
signal        shift kind                                          
momentum_12_1 0     gross 17.053   9.339 4.161 0.851   1.026   161
                    net   15.633   7.921 3.538 0.850   0.937   161
              1     gross 16.835   8.897 5.223 0.878   1.108   161
                    net   15.408   7.472 4.398 0.878   1.010   161
vol_90d       0     gross 11.168   6.241 5.211 0.508   1.241   161
                    net   10.495   5.549 4.657 0.510   1.156   161
              1     gross 10.441   4.991 4.763 0.572   1.046   161
                    net    9.771   4.323 4.140 0.572   0.974   161
z_composite   0     gross 13.731   8.019 5.736 0.605   1.281   161
                    net   12.235   6.515 4.651 0.606   1.131   161
              1     gross 14.800   8.373 5.661 0.692   1.223   161
     

C:\Users\user\AppData\Local\Temp\ipykernel_9120\600414954.py:78: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_9120\600414954.py:78: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df = pd.concat([s.rename('p'), mkt, rf], axis=1).dropna()


In [11]:
# === Diagnostic 4: does FTSE 250 membership filter close the gap? ===
import os; os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import pandas as pd, numpy as np, statsmodels.api as sm

# --- Inspect the membership file ---
mem = pd.read_parquet('data/raw/membership_snapshots.parquet')
print("Membership file columns:", list(mem.columns))
print("Sample rows:")
print(mem.head(5).to_string())
print(f"\nTotal rows: {len(mem):,}")
print(f"Unique snapshot dates: {mem['snapshot_date'].nunique() if 'snapshot_date' in mem.columns else 'N/A'}")
print(f"Unique tickers:        {mem['ticker'].nunique()}")
print(f"Unique fields:         {mem['field'].unique() if 'field' in mem.columns else 'N/A'}")

# Try to figure out the index member encoding
if 'Index Member' in mem['field'].values:
    sample = mem[mem['field'] == 'Index Member'].head(3)
    print("\nIndex Member rows sample:")
    print(sample.to_string())

# Membership tickers vs price tickers — does a suffix need adding?
mem_tickers = set(mem['ticker'].unique())
sig = pd.read_parquet('data/working/monthly_signals.parquet')
sig_tickers = set(sig['ticker'].unique())
print(f"\nMembership tickers: {len(mem_tickers)}")
print(f"Signal tickers:     {len(sig_tickers)}")
print(f"Direct overlap:     {len(mem_tickers & sig_tickers)}")
overlap_with_suffix = len({t + ' Equity' for t in mem_tickers} & sig_tickers)
print(f"Overlap if ' Equity' appended to membership: {overlap_with_suffix}")
print("\nFirst 5 membership tickers:", list(mem_tickers)[:5])
print("First 5 signal tickers:    ", list(sig_tickers)[:5])

Membership file columns: ['ticker', 'field', 'Index Member', 'Percent Weight', 'snapshot_date']
Sample rows:
      ticker              field Index Member  Percent Weight snapshot_date
0  MCX Index  INDX_MWEIGHT_HIST  1218069D LN           0.093    2010-01-29
1  MCX Index  INDX_MWEIGHT_HIST  1334987D LN           0.156    2010-01-29
2  MCX Index  INDX_MWEIGHT_HIST  1561649D LN           0.302    2010-01-29
3  MCX Index  INDX_MWEIGHT_HIST  1655637D LN           1.020    2010-01-29
4  MCX Index  INDX_MWEIGHT_HIST  1707299D LN           0.277    2010-01-29

Total rows: 48,210
Unique snapshot dates: 192
Unique tickers:        1
Unique fields:         <ArrowStringArray>
['INDX_MWEIGHT_HIST']
Length: 1, dtype: str

Membership tickers: 1
Signal tickers:     635
Direct overlap:     0
Overlap if ' Equity' appended to membership: 0

First 5 membership tickers: ['MCX Index']
First 5 signal tickers:     ['BHMU LN Equity', 'CNE LN Equity', 'JEGI LN Equity', 'RWI LN Equity', 'NXG LN Equity']


In [12]:
# === Diagnostic 4b: confirm membership column structure & ticker join ===
import os; os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import pandas as pd

mem = pd.read_parquet('data/raw/membership_snapshots.parquet')
mem['snapshot_date'] = pd.to_datetime(mem['snapshot_date'])

# Members per snapshot — should be ~250
counts = mem.groupby('snapshot_date').size()
print(f"Members per snapshot: min={counts.min()}, median={counts.median():.0f}, max={counts.max()}")
print(f"Snapshot range: {mem['snapshot_date'].min().date()}  →  {mem['snapshot_date'].max().date()}")

# Build candidate price-ticker mapping using ' Equity' suffix
mem['price_ticker'] = mem['Index Member'].astype(str).str.strip() + ' Equity'

# Check overlap with signal tickers
sig = pd.read_parquet('data/working/monthly_signals.parquet')
sig_tickers = set(sig['ticker'].unique())
mem_pt = set(mem['price_ticker'].unique())

print(f"\nUnique members ever:    {len(mem_pt)}")
print(f"Signal tickers:         {len(sig_tickers)}")
print(f"Overlap:                {len(mem_pt & sig_tickers)}")
print(f"In membership, missing in signals: {len(mem_pt - sig_tickers)}")
print(f"In signals, never a member:        {len(sig_tickers - mem_pt)}")

# Sample any non-overlap so we can see whether it's a real mismatch or whitespace
nonmatch = list(mem_pt - sig_tickers)[:5]
print(f"\nSample 'in membership, missing in signals': {nonmatch}")

# Snapshot frequency — is it monthly?
print(f"\nFirst 6 snapshot dates:")
for d in sorted(mem['snapshot_date'].unique())[:6]:
    print(f"  {pd.Timestamp(d).date()}")

Members per snapshot: min=250, median=250, max=255
Snapshot range: 2010-01-29  →  2025-12-31

Unique members ever:    641
Signal tickers:         635
Overlap:                635
In membership, missing in signals: 6
In signals, never a member:        0

Sample 'in membership, missing in signals': ['1479704D LN Equity', 'RPI LN Equity', 'SHAW LN Equity', 'PRN LN Equity', 'APN LN Equity']

First 6 snapshot dates:
  2010-01-29
  2010-02-26
  2010-03-31
  2010-04-30
  2010-05-31
  2010-06-30


In [13]:
# === Diagnostic 5: extract NB05's portfolio-formation logic ===
import os, json
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

with open('notebooks/05_backtestest_engine.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

print(f"NB05 has {len(nb['cells'])} cells.\n")

kw = ('def ', 'membership', 'snapshot', 'nlargest', 'nsmallest',
      'quantile', 'top_q', 'quintile', 'select', 'rank',
      'merge', 'isin', 'Index Member', '== 1', 'eligible',
      'universe', 'formation', 'rebalance', 'backtest')

# Show cells that look like portfolio-construction logic
for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] != 'code':
        continue
    src = ''.join(cell['source'])
    hits = [k for k in kw if k in src]
    if len(hits) >= 2:
        print(f"\n{'='*78}")
        print(f"CELL {i}  ({len(src)} chars, hits: {hits})")
        print('='*78)
        print(src[:5000])
        if len(src) > 5000:
            print(f"\n... [+{len(src)-5000} more chars]")

NB05 has 11 cells.


CELL 0  (1781 chars, hits: ['membership', 'snapshot'])
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load monthly signals
monthly = pd.read_parquet('../data/working/monthly_signals.parquet')
monthly['month'] = pd.to_datetime(monthly['month'])

# Load membership panel — tells us who's in the index each month
membership = pd.read_parquet('../data/raw/membership_panel.parquet')
membership['snapshot_date'] = pd.to_datetime(membership['snapshot_date'])
membership['month'] = membership['snapshot_date'].dt.to_period('M').dt.to_timestamp()

# Load benchmark
bench = pd.read_parquet('../data/working/benchmark_train.parquet')
bench['date'] = pd.to_datetime(bench['date'])

# Create monthly benchmark returns
bench_monthly = (bench[bench['field'] == 'TOT_RETURN_INDEX_GROSS_DVDS']
                 .sort_values('date')
                 .assign(month=lambda x: x['date'].dt.to_period('M').dt.to_timestamp())
     

In [15]:
# === Diagnostic 6: NB08 engine with NB05's per-month membership filter ===
import os; os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import pandas as pd, numpy as np, statsmodels.api as sm
from pathlib import Path

TRAIN_START, TRAIN_END = '2010-08-01', '2023-12-31'
STAMP_DUTY = 0.005

def to_ms(idx):
    return pd.PeriodIndex(pd.to_datetime(idx), freq='M').to_timestamp()

# --- 1. Build a membership panel from snapshots ---
mem_raw = pd.read_parquet('data/raw/membership_snapshots.parquet')
mem_raw['snapshot_date'] = pd.to_datetime(mem_raw['snapshot_date'])
# Map each snapshot to the month it represents (month-start)
mem_raw['mem_month'] = mem_raw['snapshot_date'].dt.to_period('M').dt.to_timestamp()
# The Index Member column holds tickers WITHOUT ' Equity' suffix
mem_raw['price_ticker'] = mem_raw['Index Member'].astype(str).str.strip() + ' Equity'

# Build month -> set of tickers
mem_panel = mem_raw[['mem_month', 'price_ticker']].drop_duplicates()
print(f"Membership panel: {len(mem_panel):,} rows, "
      f"{mem_panel['mem_month'].nunique()} months, "
      f"{mem_panel['price_ticker'].nunique()} unique tickers")

# --- 2. Load signals + costs ---
monthly = pd.read_parquet('data/working/monthly_signals.parquet')
monthly['month'] = pd.to_datetime(monthly['month'])

cs = pd.read_parquet('data/working/monthly_cs_spread.parquet')
cs['month'] = pd.to_datetime(cs['month'])

# Build z_composite if missing
if 'z_composite' not in monthly.columns:
    g = monthly.groupby('month')
    monthly['z_mom'] = g['momentum_12_1'].transform(lambda x: (x - x.mean())/x.std())
    monthly['z_vol'] = g['vol_90d'].transform(lambda x: -(x - x.mean())/x.std())
    monthly['z_composite'] = (monthly['z_mom'] + monthly['z_vol']) / 2

# --- 3. Benchmark + RF ---
b = pd.read_parquet('data/working/benchmark_train.parquet')
b['date'] = pd.to_datetime(b['date'])
b = b[b['field']=='TOT_RETURN_INDEX_GROSS_DVDS'].set_index('date')['value'].sort_index()
mkt = b.resample('ME').last().pct_change()
mkt.index = to_ms(mkt.index); mkt = mkt.rename('mkt')

s = pd.read_parquet('data/working/sonia_train.parquet')
s['date'] = pd.to_datetime(s['date'])
s = s[s['field']=='PX_LAST'].set_index('date')['value'].sort_index()
rf = s.resample('ME').mean() / 100.0 / 12.0
rf.index = to_ms(rf.index); rf = rf.rename('rf')

# --- 4. Engine, parametrised by membership filter on/off ---
def run_bt(signals, signal_col, long_top, q=0.20, extra_shift=0, apply_membership=False):
    df = signals.copy()
    if extra_shift:
        df[signal_col] = df.groupby('ticker')[signal_col].shift(extra_shift)
    df = df.dropna(subset=[signal_col, 'monthly_ret'])
    df = df[(df['month'] >= TRAIN_START) & (df['month'] <= TRAIN_END)]

    out, prev = [], set()
    for mo in sorted(df['month'].unique()):
        md = df[df['month']==mo]
        if apply_membership:
            members = set(mem_panel[mem_panel['mem_month']==mo]['price_ticker'])
            if not members:                        # fall back to nearest prior snapshot
                prior = mem_panel[mem_panel['mem_month'] <= mo]['mem_month'].max()
                members = set(mem_panel[mem_panel['mem_month']==prior]['price_ticker'])
            md = md[md['ticker'].isin(members)]
        if len(md) < 20: continue
        if long_top:
            thr = md[signal_col].quantile(1-q); sel = md[md[signal_col] >= thr]
        else:
            thr = md[signal_col].quantile(q);   sel = md[md[signal_col] <= thr]
        if sel.empty: continue
        cur = set(sel['ticker'])
        gross = sel['monthly_ret'].mean()
        buys, sells = cur - prev, prev - cur
        msp = cs[cs['month']==mo].set_index('ticker')['spread']
        def avs(t):
            if not t: return 0.0
            f = msp.reindex(list(t)).dropna()
            return f.mean() if len(f) else (msp.mean() if len(msp) else 0.0)
        n = len(cur); bw, sw = len(buys)/n, len(sells)/n
        hb = avs(buys)/2  if buys  else 0
        hs = avs(sells)/2 if sells else 0
        tc = bw*(hb + STAMP_DUTY) + sw*hs
        out.append({'month': mo, 'gross': gross, 'net': gross-tc, 'tc': tc, 'n': n})
        prev = cur
    bt = pd.DataFrame(out).set_index('month')
    if not bt.empty:
        bt.index = to_ms(bt.index)
    return bt

def alpha_nw(series, lags=6):
    s = series.copy(); s.index = to_ms(s.index)
    df = pd.concat([s.rename('p'), mkt, rf], axis=1, sort=True).dropna()
    if len(df) < 10:
        return {'alpha%': np.nan, 't': np.nan, 'beta': np.nan,
                'sharpe': np.nan, 'mean%': np.nan, 'n_mo': len(df)}
    y = df['p'] - df['rf']; X = sm.add_constant(df['mkt'] - df['rf'])
    m = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return {'alpha%': m.params['const']*12*100, 't': m.tvalues['const'],
            'beta': m.params.iloc[1],
            'sharpe': (y.mean()/y.std())*np.sqrt(12),
            'mean%': df['p'].mean()*12*100, 'n_mo': len(df)}

# --- 5. Run grid: 4 cells per signal (shift=0/1 × membership=off/on) ---
configs = [('momentum_12_1', True), ('vol_90d', False), ('z_composite', True)]

rows = []
for sig, lt in configs:
    if sig not in monthly.columns:
        print(f"  ! {sig} missing"); continue
    for shift in (0, 1):
        for mem_filter in (False, True):
            bt = run_bt(monthly, sig, long_top=lt, q=0.20,
                        extra_shift=shift, apply_membership=mem_filter)
            if bt.empty:
                continue
            tag = f"shift={shift} mem={'Y' if mem_filter else 'N'}"
            print(f"  {sig:15s} {tag:18s} {len(bt)} mo, avg n={bt['n'].mean():.1f}")
            for kind in ('gross', 'net'):
                r = alpha_nw(bt[kind])
                r.update({'signal': sig, 'shift': shift,
                          'membership': mem_filter, 'kind': kind})
                rows.append(r)

out = pd.DataFrame(rows).set_index(['signal','shift','membership','kind'])
out = out[['mean%', 'alpha%', 't', 'beta', 'sharpe', 'n_mo']]
pd.set_option('display.float_format', '{:.3f}'.format)
print("\n=== shift × membership grid ===")
print(out)

Membership panel: 48,210 rows, 192 months, 641 unique tickers
  momentum_12_1   shift=0 mem=N      161 mo, avg n=89.6
  momentum_12_1   shift=0 mem=Y      161 mo, avg n=49.5
  momentum_12_1   shift=1 mem=N      161 mo, avg n=89.4
  momentum_12_1   shift=1 mem=Y      161 mo, avg n=49.3
  vol_90d         shift=0 mem=N      161 mo, avg n=91.8
  vol_90d         shift=0 mem=Y      161 mo, avg n=50.3
  vol_90d         shift=1 mem=N      161 mo, avg n=91.6
  vol_90d         shift=1 mem=Y      161 mo, avg n=50.3
  z_composite     shift=0 mem=N      161 mo, avg n=89.6
  z_composite     shift=0 mem=Y      161 mo, avg n=49.5
  z_composite     shift=1 mem=N      161 mo, avg n=89.3
  z_composite     shift=1 mem=Y      161 mo, avg n=49.3

=== shift × membership grid ===
                                      mean%  alpha%     t  beta  sharpe  n_mo
signal        shift membership kind                                          
momentum_12_1 0     False      gross 17.053   9.339 4.161 0.851   1.026   161

In [16]:
# === Save reconciliation artefacts ===
import os; os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
from pathlib import Path
import pandas as pd

# Save the grid (assuming `out` is still in memory from the last cell)
out_path = Path('data/reconciliation_grid.csv')
out.to_csv(out_path)
print(f"Saved: {out_path}")

# Build a focused canonical-vs-original comparison table
canon = (out.xs(('shift', 1, True), level=('shift', 'membership'))
            if False else None)  # unused; readable layout below

# Cleaner: pivot to a one-page summary
summary = (out.reset_index()
           .pivot_table(index=['signal','kind'],
                        columns=['shift','membership'],
                        values=['alpha%','t']))
summary.to_csv('data/reconciliation_summary.csv')
print("Saved: reconciliation_summary.csv")

# Also save the raw monthly returns for the canonical engine, all three signals,
# so NB09 (factor attribution) and the dissertation can pull from them later.
# Re-run only the four canonical configs and persist.

print("\n=== Canonical engine (shift=1, membership=Y) — saving monthly returns ===")
import numpy as np
canonical_runs = {}
for sig, lt in [('momentum_12_1', True), ('vol_90d', False), ('z_composite', True)]:
    bt = run_bt(monthly, sig, long_top=lt, q=0.20,
                extra_shift=1, apply_membership=True)
    bt['signal'] = sig
    canonical_runs[sig] = bt
    fn = f"data/working/canonical_{sig}.parquet"
    bt.reset_index().to_parquet(fn)
    print(f"  saved {fn}  ({len(bt)} months)")

Saved: data\reconciliation_grid.csv
Saved: reconciliation_summary.csv

=== Canonical engine (shift=1, membership=Y) — saving monthly returns ===
  saved data/working/canonical_momentum_12_1.parquet  (161 months)
  saved data/working/canonical_vol_90d.parquet  (161 months)
  saved data/working/canonical_z_composite.parquet  (161 months)


In [17]:
# === Canonical headline numbers (training period) ===
import os; os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import pandas as pd, numpy as np, statsmodels.api as sm
from pathlib import Path

def to_ms(idx):
    return pd.PeriodIndex(pd.to_datetime(idx), freq='M').to_timestamp()

# Load benchmark & RF
b = pd.read_parquet('data/working/benchmark_train.parquet')
b['date'] = pd.to_datetime(b['date'])
b = b[b['field']=='TOT_RETURN_INDEX_GROSS_DVDS'].set_index('date')['value'].sort_index()
mkt = b.resample('ME').last().pct_change(); mkt.index = to_ms(mkt.index); mkt.name='mkt'

s = pd.read_parquet('data/working/sonia_train.parquet')
s['date'] = pd.to_datetime(s['date'])
s = s[s['field']=='PX_LAST'].set_index('date')['value'].sort_index()
rf = s.resample('ME').mean()/100/12; rf.index = to_ms(rf.index); rf.name='rf'

def metrics(port_net, label, lags=6):
    s = port_net.copy(); s.index = to_ms(s.index)
    df = pd.concat([s.rename('p'), mkt, rf], axis=1, sort=True).dropna()
    y = df['p'] - df['rf']
    X = sm.add_constant(df['mkt'] - df['rf'])
    m = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    cum = (1 + df['p']).cumprod()
    dd  = (cum / cum.cummax() - 1).min()
    bench_ann = (1 + df['mkt']).prod()**(12/len(df)) - 1
    port_ann  = (1 + df['p']).prod()**(12/len(df)) - 1
    return {
        'label':         label,
        'months':        len(df),
        'ann_ret_%':     port_ann * 100,
        'bench_ann_%':   bench_ann * 100,
        'ann_vol_%':     df['p'].std() * np.sqrt(12) * 100,
        'sharpe':        (y.mean()/y.std()) * np.sqrt(12),
        'capm_alpha_%':  m.params['const'] * 12 * 100,
        't_alpha_NW6':   m.tvalues['const'],
        'beta':          m.params.iloc[1],
        'max_dd_%':      dd * 100,
        'calmar':        port_ann / abs(dd) if dd != 0 else np.nan,
    }

rows = []
for sig, label in [('momentum_12_1', 'Momentum'),
                   ('vol_90d',       'Low-Vol'),
                   ('z_composite',   'Combined')]:
    df = pd.read_parquet(f'data/working/canonical_{sig}.parquet')
    df['month'] = pd.to_datetime(df['month'])
    df = df.set_index('month')
    rows.append(metrics(df['net'], label))

summary = pd.DataFrame(rows).set_index('label')
pd.set_option('display.float_format', '{:.3f}'.format)
print("\n=== Canonical engine: training period (2010-08 to 2023-12) ===")
print("(net of Corwin-Schultz spreads + 0.5% stamp duty on buys)")
print()
print(summary.to_string())

# Save it
summary.to_csv('data/canonical_training_summary.csv')
print("\nSaved: data/canonical_training_summary.csv")


=== Canonical engine: training period (2010-08 to 2023-12) ===
(net of Corwin-Schultz spreads + 0.5% stamp duty on buys)

          months  ann_ret_%  bench_ann_%  ann_vol_%  sharpe  capm_alpha_%  t_alpha_NW6  beta  max_dd_%  calmar
label                                                                                                         
Momentum     161     12.842        8.078     15.511   0.809         4.925        2.638 0.940   -31.399   0.409
Low-Vol      161      8.147        8.078      9.796   0.768         2.529        2.535 0.616   -20.026   0.407
Combined     161     12.256        8.078     12.154   0.949         5.579        3.519 0.736   -24.525   0.500

Saved: data/canonical_training_summary.csv


In [18]:
import json
with open('notebooks/06_holdout_test.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)
for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] != 'code': continue
    src = ''.join(cell['source'])
    if 'membership' in src.lower() or 'isin(' in src:
        print(f"=== Cell {i} ===")
        print(src[:1500])
        print()

=== Cell 0 ===
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("=== UNLOCKING HOLDOUT: 2024-2025 ===")
print("This is a ONE-TIME test. Results are final.\n")

# Load holdout
holdout = pd.read_parquet('../data/holdout_locked/equities_holdout.parquet')
holdout['date'] = pd.to_datetime(holdout['date'])
print(f"Holdout: {len(holdout):,} rows, {holdout['ticker'].nunique()} tickers")
print(f"Date range: {holdout['date'].min().date()} to {holdout['date'].max().date()}")

# Load training data (need trailing history for signal computation)
train = pd.read_parquet('../data/working/equities_train.parquet')
train['date'] = pd.to_datetime(train['date'])

# Combine train + holdout for signal computation
# (signals at Jan 2024 need data from 2023 for lookbacks)
full = pd.concat([train, holdout], ignore_index=True)
full = full.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f"\nFull panel: {len(full):,} rows")

# Mem